# Modelling 
## Strategie 1
For this time, we'll no try to predict the `global` target. Like we have seen in our EDA\
the high correlation between the averages and the feature involved the `home_team_goal` & `away_team_goal`.
- Target = **For classification problem we gonna cut in bins the goal team, it might be the scored goal or the conceded goal**
- Features = **goal averages for team at home and away before the current match**

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

In [ ]:
path = Path.cwd().parent.parent / 'data' / '2022_23' / 'results' / 'ligue1.csv'
df = pd.read_csv(path, parse_dates=True, index_col='date_time')
df.head()

### Feature Engineering

In [ ]:
def feature_engineering(df):
    """This function aims to create new features
    It will calculate for each team
    the average of goal scored and conceded before the current match
    """
    data = df.copy()
    data = data.sort_values(by='journ?e', ascending=True)
    data = data[['journ?e', 'home_team', 'away_team', 'total_home_team_goal',
                     'total_away_team_goal', 'global']]
    for row in data.iterrows():
        data['ht_scored_avg'] = data.groupby('home_team')['total_home_team_goal'].transform(lambda x: x.expanding().mean().shift())
        data['ht_conceded_avg'] = data.groupby('home_team')['total_away_team_goal'].transform(lambda x: x.expanding().mean().shift())
        data['at_scored_avg'] = data.groupby('away_team')['total_away_team_goal'].transform(lambda x: x.expanding().mean().shift())
        data['at_conceded_avg'] = data.groupby('away_team')['total_home_team_goal'].transform(lambda x: x.expanding().mean().shift())
    return data

In [ ]:
transformed_df = feature_engineering(df)

In [ ]:
transformed_df.tail()

In [ ]:
# we gonna drop the missing values
data = transformed_df[transformed_df['journ?e']>2]

### Preprocessing

In [ ]:
data['class'] = pd.cut(x=data['total_home_team_goal'], bins=[min(data['total_home_team_goal']), 1,
                                                    max(data['total_home_team_goal'])],
                                                    labels=['under 1.5','over 1.5'], include_lowest=True)

In [ ]:
data['class'].value_counts()

In [ ]:
code = {
        'under 1.5': 0,
        'over 1.5': 1}
data['class'] = data['class'].map(code)

In [ ]:
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV, learning_curve
from sklearn.ensemble import RandomForestClassifier, VotingClassifier, RandomForestRegressor
from sklearn.preprocessing import LabelEncoder, MinMaxScaler, StandardScaler, RobustScaler
from sklearn.pipeline import make_pipeline
from sklearn.compose import make_column_transformer
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.linear_model import RidgeClassifier, LinearRegression

In [ ]:
# X represent the feature used to make prediction
# y the target to predict
X = data[['ht_scored_avg', 'at_conceded_avg']]
y = data['class']
# We gonna split the data into train and test set
x_train, x_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=33)

In [ ]:
def evaluation(x_test, y_test, model):
    """Test our model on the test set
    and show some metrics to evaluate performance of the model
    """
    print(f"Final Score {model.__class__.__name__} : {model.score(x_test, y_test)}")
    print(confusion_matrix(y_test, model.predict(x_test)))
    print(classification_report(y_test, model.predict(x_test)))

    N, train_score, val_score = learning_curve(
        model, x_train, y_train, train_sizes= np.linspace(0.1, 1.0, 10), cv=5
    )

    plt.plot(N, train_score.mean(axis=1), label='train')
    plt.plot(N, val_score.mean(axis=1), label='validation')
    plt.xlabel('train size')
    plt.legend()
    plt.show()

In [ ]:
from sklearn.linear_model import RidgeClassifier
ridge_pipeline = make_pipeline(StandardScaler(), RidgeClassifier())
ridge_pipeline.fit(x_train, y_train)

In [ ]:
evaluation(x_test, y_test, model=ridge_pipeline)

### Interpretation
It seems like, our data does not represent well the problem, thus the score is pretty low cause we facing a underfitting issue. Though, the goal averages can serve later.

## Strategie 2
In football, there's something we call `tendance` which represent the form of the team, Thus we can observe for a time, a team scoring and winning much, but in a certain time the same team can lose all of their matchs.\
So if we want to predict the goal scored by the home team, we have to find the last five goal scored by the team and give them to our model

#### Feature Engineering 

In [ ]:
data = df.copy()
data.sort_values(by='journ?e', ascending=True, inplace=True)
the_list = []
for i, row in data.iterrows():
    the_dict = {}
    last_values_home_team_scored = data[((data['home_team']== row['home_team']) | (data['away_team']== row['home_team'])) & (data['journ?e'] < row['journ?e'])]['total_home_team_goal'].tail().tolist()
    last_values_away_team_scored = data[((data['away_team']== row['away_team']) | (data['home_team']== row['away_team'])) & (data['journ?e'] < row['journ?e'])]['total_home_team_goal'].tail().to_list()
    the_dict['journ?e'], the_dict['home_team'], the_dict['away_team'], the_dict['home_team_goal'], the_dict['away_team_goal']\
     = row['journ?e'], row['home_team'], row['away_team'], row['total_home_team_goal'], row['total_away_team_goal']
    the_dict['lastValue_home_team_scored'] = last_values_home_team_scored
    the_dict['lastValue_away_team_scored'] = last_values_away_team_scored
    the_list.append(the_dict)
data = pd.DataFrame(the_list)

In [ ]:
data.head(60)

In [ ]:
data['home_team_goal_class'] = pd.cut(x=data['home_team_goal'], bins=[min(data['home_team_goal']), 1, max(
    data['home_team_goal'])], labels=['under 1.5', 'over 1.5'], include_lowest=True)
data['away_team_goal_class'] = pd.cut(x=data['away_team_goal'], bins=[min(data['away_team_goal']), 1, max(
    data['away_team_goal'])], labels=['under 1.5', 'over 1.5'], include_lowest=True)

In [ ]:
def spread_list(data, col_prefix='lastValueN'):
    home_values = data['lastValue_home_team_scored']
    for i, value in enumerate(home_values):
        data[f"home_{col_prefix}{i+1}"] = value
    away_values = data['lastValue_away_team_scored']
    for i, value in enumerate(away_values):
        data[f"away_{col_prefix}{i+1}"] = value
    return data

data = data.apply(spread_list, axis=1)
short_data = data.drop(['lastValue_home_team_scored', 'lastValue_away_team_scored'], axis=1)

In [ ]:
short_data = short_data[short_data['journ?e']>5]

In [ ]:
short_data[['home_lastValueN1', 'home_lastValueN2', 'home_lastValueN3', 'home_lastValueN4', 'home_lastValueN5']] =\
short_data[['home_lastValueN1', 'home_lastValueN2', 'home_lastValueN3', 'home_lastValueN4', 'home_lastValueN5']].astype(int)

In [ ]:
short_data['home_team_goal_class'].value_counts()

In [ ]:
code = {
        'under 1.5': 0,
        'over 1.5': 1}
short_data['home_team_goal_class'] = short_data['home_team_goal_class'].map(code)

In [ ]:
short_data.info()

In [ ]:
X = short_data[['home_lastValueN1', 'home_lastValueN2', 'home_lastValueN3', 'home_lastValueN4', 'home_lastValueN5']]
y = short_data['home_team_goal_class']
x_train, x_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=22)

In [ ]:
from sklearn.linear_model import RidgeClassifier
ridge_estimator = make_pipeline(MinMaxScaler(), RidgeClassifier())
ridge_estimator.fit(x_train, y_train)

In [ ]:
evaluation(x_test, y_test, model=ridge_estimator)

In [ ]:
from sklearn.ensemble import RandomForestClassifier
rfc_estimator = make_pipeline(MinMaxScaler(), RandomForestClassifier())
rfc_estimator.fit(x_train, y_train)

In [ ]:
evaluation(x_test, y_test, model=rfc_estimator)

### Interpretation
We have seen that our different model are unable to predict with accuracy the `over 1.5` class

In [ ]:
from sklearn.ensemble import VotingClassifier
clf1 = RidgeClassifier()
clf2 = RandomForestClassifier()
eclf = VotingClassifier(
    estimators=[('ridge', clf1), ('rfc', clf2)], voting='hard'
)
for clf, label in zip([clf1, clf2, eclf], ['Ridge Classifier', 'Random Forest Classifier', 'ensemble']):
    scores = cross_val_score(clf, X, y, scoring='accuracy', cv=5)
    print("Accuracy: %0.2f (+/- %0.2f) [%s]" % (scores.mean(), scores.std(), label))

In [ ]:
from joblib import dump, load
dump(ridge_estimator, '../machine_learning/models/classifier/ridge_clf.joblib')
dump(ridge_estimator.named_steps['minmaxscaler'], '../machine_learning/models/classifier/minmaxscaler_ridge.joblib')